In [5]:
# 2. استيراد المكتبات
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import os

In [6]:
# 3. تحديد المسار
MODEL_PATH = "E:/project2026/projectfood/AraSenttt/models/sentiment_model"
DATA_PATH = "sentiment_data.csv"  # تأكد أن الملف في نفس المجلد
os.makedirs(MODEL_PATH, exist_ok=True)

In [7]:
# 4. قراءة البيانات
print("📊 جاري قراءة البيانات...")
df = pd.read_csv(DATA_PATH)
print(f"عدد العينات: {len(df)}")
print("توزيع المشاعر:")
print(df['sentiment'].value_counts())

📊 جاري قراءة البيانات...
عدد العينات: 30
توزيع المشاعر:
sentiment
positive    12
negative    11
neutral      7
Name: count, dtype: int64


In [8]:
# 5. تحويل المشاعر لأرقام
sentiment_map = {'positive': 0, 'negative': 1, 'neutral': 2}
df['label'] = df['sentiment'].map(sentiment_map)

In [9]:
# 6. تقسيم البيانات
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print(f"\nتدريب: {len(train_df)} عينة")
print(f"تحقق: {len(val_df)} عينة")


تدريب: 24 عينة
تحقق: 6 عينة


In [1]:
# 7. تحميل tokenizer والنموذج
print("\n📥 جاري تحميل النموذج...")
tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")
model = AutoModelForSequenceClassification.from_pretrained(
    "aubmindlab/bert-base-arabertv2", 
    num_labels=3
)
model.config.id2label = {0: 'positive', 1: 'negative', 2: 'neutral'}
model.config.label2id = {'positive': 0, 'negative': 1, 'neutral': 2}


📥 جاري تحميل النموذج...


NameError: name 'AutoTokenizer' is not defined

In [ ]:
# 8. تجهيز البيانات
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
val_dataset = Dataset.from_pandas(val_df[['text', 'label']])

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

In [ ]:
# 9. دالة التقييم
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

In [ ]:
# 10. إعداد التدريب
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,  # زيادة عدد المرات لتعلم أفضل
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

In [ ]:
# 11. إنشاء Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
# 12. التدريب
print("\n🚀 بدء التدريب...")
print("="*50)
trainer.train()

In [ ]:
# 13. حفظ النموذج
print("\n💾 جاري حفظ النموذج...")
model.save_pretrained(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)
print(f"✅ تم الحفظ في: {MODEL_PATH}")

In [ ]:
# 14. تقييم نهائي
print("\n📊 نتائج التقييم:")
results = trainer.evaluate()
for key, value in results.items():
    print(f"{key}: {value:.4f}")


In [ ]:
# 15. اختبار النموذج
print("\n🧪 اختبار النموذج:")
print("="*50)
test_texts = [
    "الاكل لذيذ جداً",
    "الخدمة سيئة",
    "السعر متوسط",
    "المطعم نظيف ومرتب",
    "التوصيل جاء متأخر جداً",
    "الاسعار غالية شوي",
    "العاملين محترمين"
]

for text in test_texts:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.softmax(outputs.logits, dim=1)
        pred_class = torch.argmax(predictions, dim=1).item()
    
    sentiment = {0: 'إيجابي', 1: 'سلبي', 2: 'محايد'}[pred_class]
    confidence = predictions[0][pred_class].item()
    print(f"\n📝 {text}")
    print(f"😊 {sentiment} (ثقة: {confidence:.1%})")